RAG Chatbot

install packages

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers chromadb pypdf fpdf2

  Using cached sentence_transformers-5.6.1-py3-none-any.whl (596 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)
  Using cached pypdf-6.14.2-py3-none-any.whl (349 kB)
  Using cached fpdf2-2.8.7-py3-none-any.whl (327 kB)
     ---------------------------------------- 0.0/150.9 kB ? eta -:--:--
     ----- ------------------------------- 20.5/150.9 kB 682.7 kB/s eta 0:00:01
     ---------- -------------------------- 41.0/150.9 kB 495.5 kB/s eta 0:00:01
     ---------- -------------------------- 41.0/150.9 kB 495.5 kB/s eta 0:00:01
     ---------- -------------------------- 41.0/150.9 kB 495.5 kB/s eta 0:00:01
     --------------- --------------------- 61.4/150.9 kB 233.8 kB/s eta 0:00:01
     ---------------------- -------------- 92.2/150.9 kB 309.1 kB/s eta 0:00:01
     ----------------------------- ------ 122.9/150.9 kB 379.3 kB/s eta 0:00:01
     ------------------------------------ 150.9/150.9 kB 346.5 kB/s eta 0:00:00
     ---------------------------------------- 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


imports

In [1]:
import os
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader
from fpdf import FPDF

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


groq api key

In [ ]:
# set api key
os.environ["GROQ_API_KEY"] = "your key here"
client = Groq()

knowledge base text

In [3]:
knowledge_text = """Renewable Energy Sources

Renewable energy comes from natural sources that replenish themselves faster than they are consumed. The main types are solar, wind, hydro, geothermal, and biomass energy. Unlike fossil fuels, these sources produce little to no greenhouse gas emissions during operation.

Solar energy is captured using photovoltaic cells that convert sunlight directly into electricity. Solar panels can be installed on rooftops or in large solar farms. The efficiency of solar panels has improved significantly over the past decade, making solar one of the cheapest sources of new electricity generation in many regions.

Wind energy is generated using turbines that convert the kinetic energy of moving air into electricity. Wind farms can be built onshore or offshore. Offshore wind farms generally produce more consistent power because wind speeds over water are higher and steadier than over land.

Hydropower is produced by using flowing or falling water to spin turbines connected to generators. Large dams can generate substantial amounts of electricity, but they can also have significant environmental impacts on river ecosystems and local communities.

Geothermal energy uses heat from within the earth to generate electricity or provide direct heating. This source is most effective in regions with high volcanic or tectonic activity, where hot water and steam are close to the surface.

Biomass energy is produced by burning organic materials such as wood, crop residues, or animal waste. While biomass is renewable, its environmental benefit depends on how sustainably the source material is harvested and replaced.

The transition to renewable energy is driven by the need to reduce carbon emissions, improve energy security, and lower long term energy costs. Many countries have set targets to increase the share of renewables in their energy mix over the coming decades."""

create pdf

In [4]:
# write knowledge base to pdf
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
for line in knowledge_text.split("\n"):
    if line.strip() == "":
        pdf.ln(8)
    else:
        pdf.multi_cell(0, 8, line)
pdf.output("knowledge_base.pdf")

C:\Users\HP\AppData\Local\Temp\ipykernel_13668\3679572613.py:4: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)


load pdf

In [5]:
# extract text from pdf
reader = PdfReader("knowledge_base.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text()

len(text)

1890

chunking

In [6]:
# split text into overlapping chunks
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(text)
len(chunks)

8

embeddings

In [7]:
# load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# generate embeddings
embeddings = embed_model.encode(chunks).tolist()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1948.45it/s]


vector db

In [8]:
# create chroma collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="knowledge_base")

# store chunks and embeddings
ids = [str(i) for i in range(len(chunks))]
collection.add(documents=chunks, embeddings=embeddings, ids=ids)

similarity search

In [9]:
# retrieve top matching chunks
def retrieve_chunks(query, top_k=3):
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    return results["documents"][0]

rag pipeline

In [10]:
# build prompt with context and generate answer
def rag_answer(query):
    retrieved = retrieve_chunks(query)
    context = "\n".join(retrieved)

    prompt = f"""context:
{context}

question: {query}
answer using only the context above."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        top_p=0.9
    )
    return response.choices[0].message.content

test queries

In [11]:
print(rag_answer("what is offshore wind energy and why is it more consistent"))

Offshore wind energy is generated using turbines in wind farms built offshore. It is more consistent because wind speeds over water are higher and steadier than over land.


In [12]:
print(rag_answer("what factors affect the environmental benefit of biomass energy"))

The environmental benefit of biomass energy depends on how sustainably the source material is harvested and replaced.
